# SAM 2 smoke run
Run top to bottom on a GPU runtime. Cells: mount Drive, set the data root, get the repo, install, download checkpoints, build first-frame box prompts for one DAVIS video, run SAM 2.1 Tiny and Large, list the outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT = ""  # @param {type:"string"}
assert DATA_ROOT, "Fill in DATA_ROOT (the Drive folder that holds the datasets)"
assert os.path.exists(DATA_ROOT), f"DATA_ROOT does not exist: {DATA_ROOT}"
os.environ["SAM2FR_DATA_ROOT"] = DATA_ROOT

In [ ]:
BRANCH = "phase-0b-sam2"  # @param {type:"string"}  (after this phase is merged, set it to main)
REPO_URL = "https://github.com/VEDITHREDDY26/sam2-failure-recovery.git"
REPO_DIR = "/content/sam2-failure-recovery"
import os
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch origin
else:
    !git clone {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} checkout {BRANCH}
!git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

In [ ]:
# Colab ships plain, headless and contrib OpenCV builds that share one cv2 folder;
# remove all three, then install only the pinned contrib build.
!pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python
!pip install -q -r requirements-colab.txt
!pip install -q --force-reinstall --no-deps opencv-contrib-python==4.11.0.86
!pip list 2>/dev/null | grep -i opencv

Restart the session now (Runtime > Restart session), then re-run the Drive, DATA_ROOT and repo cells and continue from the check cell below; skip the install cell. If `cv2` has no version or `TrackerCSRT_create` prints False, the install cell did not finish: run it again, then restart.

In [ ]:
import platform, numpy, cv2, torch
print("python", platform.python_version())
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
print("numpy", numpy.__version__)
print("cv2", cv2.__version__)
print("TrackerCSRT_create", hasattr(cv2, "TrackerCSRT_create"))

In [ ]:
!bash scripts/get_checkpoints.sh

In [ ]:
VIDEO = "bear"  # @param {type:"string"}
import json, sys
from pathlib import Path
sys.path.insert(0, ".")
from src.common.boxes import box_from_mask
from src.common.masks import label_map_to_masks, read_label_map
from src.sam2_runner.smoke import load_data_config

cfg = load_data_config()
# The first-frame annotation is read here, in the notebook, to build the initial prompt.
# src/sam2_runner itself never reads ground truth.
annotation = cfg.data_root / cfg.davis.root / "Annotations" / cfg.davis.resolution / VIDEO / "00000.png"
boxes = {}
for obj_id, mask in label_map_to_masks(read_label_map(annotation)).items():
    box = box_from_mask(mask)
    if box is not None:
        boxes[str(obj_id)] = [box.x0, box.y0, box.x1, box.y1]
PROMPTS = cfg.data_root / cfg.output_root / "prompts" / f"{VIDEO}.json"
PROMPTS.parent.mkdir(parents=True, exist_ok=True)
PROMPTS.write_text(json.dumps({"video": VIDEO, "frame_index": 0, "boxes": boxes}))
print(PROMPTS, boxes)

In [ ]:
!python -m src.sam2_runner.smoke --model tiny --video {VIDEO} --prompts {PROMPTS}

In [ ]:
!python -m src.sam2_runner.smoke --model large --video {VIDEO} --prompts {PROMPTS}

In [ ]:
import pandas as pd, torch
from src.common.layout import masks_dir, signals_path
from src.common.masks import label_map_to_masks, read_label_map

print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
frames = sorted((cfg.data_root / cfg.davis.root / "JPEGImages" / cfg.davis.resolution / VIDEO).glob("*.jpg"))
out_root = cfg.data_root / cfg.output_root
for model in ("tiny", "large"):
    method = f"sam2_{model}_smoke"
    pngs = sorted(masks_dir(out_root, method, "davis_val", VIDEO).glob("*.png"))
    ids = sorted({i for p in pngs for i in label_map_to_masks(read_label_map(p))})
    print(f"{model}: {len(frames)} frames, {len(pngs)} PNGs, object ids present {ids}")
    print(pd.read_csv(signals_path(out_root, method, "davis_val", VIDEO)).head())

## Object-score source check (placeholder)
Filled in by the next task: confirm where SAM 2 exposes the object score and compare it with the mean-mask-logit fallback.

In [ ]:
# placeholder: object-score source check (next task)